In [7]:
import pickle
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
from scipy.spatial.transform import Rotation as R

from frankar3_control.franka_r3_robot import FR3_ROBOT
from scipy.optimize import least_squares
# from data_processing.utils import estimate_payload_with_bias, compensate_weight, optimize_payload_with_bias

In [8]:
saved_sample_path = "/home/jiuzl/robomimic_suite/robot_camera_control/data_processing/calib_data_samples.pkl"
validation_sample_path = "/home/jiuzl/robomimic_suite/robot_camera_control/data_processing/validation_check_samples.pkl"

In [9]:
@dataclass
class PoseSample:
    name: str
    forces: np.ndarray  # (N, 3)
    torques: np.ndarray  # (N, 3)
    orientation: R      # Rotation from  sensor to base frame
    pose: np.ndarray    # Affine matrix of EE pose in world frame


In [10]:
with open(saved_sample_path, 'rb') as f:
    calib_samples = pickle.load(f)
with open(validation_sample_path, 'rb') as f:
    validation_samples = pickle.load(f)

In [11]:
# combine calib and validation samples for better estimation
all_samples = calib_samples + validation_samples

In [15]:
def optimize_payload(data, initial_guess, g=9.81):
    """
    Refine payload estimation using non-linear optimization.

    Parameters
    ----------
    data : list of dict
        Each element must contain:
        {
          'R_sw': 3x3 rotation matrix (sensor -> world),
          'force':  length-3 ndarray (sensor frame),
          'torque': length-3 ndarray (sensor frame)
        }

    initial_guess : dict
        Output from estimate_payload_with_bias() to initialize optimization.

    g : float
        Gravity magnitude (default 9.81)

    Returns
    -------
    result : dict
        Same format as estimate_payload_with_bias(), but refined.
    """
    

    def residuals(params):
        m = params[0]
        r = params[1:4]

        res = []
        for sample in data:
            R_sw = sample['R_sw'] # 3 by 3 rotation from sensor to world
            f_meas = sample['force'] # n by 3
            tau_meas = sample['torque'] # n by 3

            g_s = R_sw.T @ np.array([0.0, 0.0, -g])
            f_pred = m * g_s
            tau_pred = np.cross(r, m * g_s)

            res.extend((f_meas - f_pred).flatten())
            res.extend((tau_meas - tau_pred).flatten())

        return np.array(res)

    x0 = np.hstack([
        initial_guess['mass'],
        initial_guess['com']
    ])

    result = least_squares(residuals, x0, verbose=2)
    m_opt = result.x[0]
    r_opt = result.x[1:4]

    if m_opt <= 0:
        raise ValueError("Optimized mass is non-positive. Check data quality.")

    return {
        'mass': m_opt,
        'com': r_opt,
        'optimization_success': result.success,
        'optimization_message': result.message
    }

In [20]:
def estimate_mass_and_com(samples: List[PoseSample]) -> Dict[str, np.ndarray]:
    """Estimate mass and COM using nonlinear optimization with bounds."""
    data = []
    for i in range(len(samples)):
        sample = samples[i]
        R_sw = sample.orientation.as_matrix()  # sensor to world rotation
        # forces = sample.forces
        # torques = sample.torques
        mean_forces = np.mean(sample.forces, axis=0)
        mean_torques = np.mean(sample.torques, axis=0)
        data.append({'R_sw': R_sw, 'force': mean_forces, 'torque': mean_torques})
    
        init_guess_mass = 0.2
        init_guess_com = np.array([0.0, 0.0, 0.2])
        init_guess = {
            'mass': init_guess_mass,
            'com': init_guess_com}
    result = optimize_payload(data, init_guess, g=9.81)

    return result

In [21]:
result = estimate_mass_and_com(all_samples)
mass_kg = result['mass']
com_m = result['com']
print("\nEstimated gripper parameters:")
print(f"  Mass: {mass_kg:.3f} kg")
print(f"  COM (stiffness frame): [{com_m[0]:.4f}, {com_m[1]:.4f}, {com_m[2]:.4f}] m")
print("\nSet these values in your controller/load parameters so external wrench at the stiffness frame trends to zero at static.")
    

   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         1.2475e+01                                    4.10e+01    
       1              2         1.1776e+01      6.98e-01       9.45e-02       4.27e-01    
       2              3         1.1774e+01      2.67e-03       1.35e-02       3.86e-08    
       3              4         1.1774e+01      0.00e+00       0.00e+00       3.86e-08    
`xtol` termination condition is satisfied.
Function evaluations 4, initial cost 1.2475e+01, final cost 1.1774e+01, first-order optimality 3.86e-08.

Estimated gripper parameters:
  Mass: 0.174 kg
  COM (stiffness frame): [-0.0120, -0.0805, 0.1347] m

Set these values in your controller/load parameters so external wrench at the stiffness frame trends to zero at static.


In [22]:
ee_pose_in_flange_frame = np.array([[ 0.70710677,  0.70710677,  0.,  0.        ],
 [-0.70710677,  0.70710677,  0.,  0.        ],
 [ 0.,  0.,  1.,  0.1034    ],
 [ 0.,  0.,  0.,  1.        ]])

com_load_in_ee_frame = com_m
com_load_in_flange_frame = ee_pose_in_flange_frame[:3,:3] @ com_load_in_ee_frame + ee_pose_in_flange_frame[:3,3]
print(f"COM of load in Flange Frame: [{com_load_in_flange_frame[0]:.4f}, {com_load_in_flange_frame[1]:.4f}, {com_load_in_flange_frame[2]:.4f}] m")

COM of load in Flange Frame: [-0.0654, -0.0484, 0.2381] m
